# 🛡️ Phishing Detection Model using BERT

This notebook contains a fine-tuned **BERT-based binary text classification model** designed to detect phishing URLs or text patterns. The model classifies inputs as either **"Safe"** or **"Not Safe"** based on training data containing labeled examples.

📚 **Source/Reference**:  
The model and training pipeline were created by following the YouTube tutorial:  
**[Fine-Tune BERT for Text Classification with HuggingFace 🤗](https://www.youtube.com/watch?v=4QHg8Ix8WWQ)**  
by *AI Engineering*.

The notebook includes:
- Tokenization and preprocessing with Hugging Face `transformers` and `datasets`
- Model loading (`AutoModelForSequenceClassification`)
- Training using `Trainer`
- Evaluation with accuracy and ROC AUC metrics
- Model saving and inference with `pipeline`

> 🚀 This notebook serves as a practical guide for anyone looking to build a phishing detection system using NLP and transfer learning.


## import libraries and dataset

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 21.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.8 MB/s eta 0:00:00


In [ ]:
from datasets import DatasetDict, Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

In [ ]:
ddict = load_dataset("shawhin/phishing-site-classification")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/98.0k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/21.4k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/450 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/450 [00:00<?, ? examples/s]

In [ ]:
ddict

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
})

## Inizialize Model and Tokenizer

In [ ]:
model_path = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_path)

id2label = {0: "Safe", 1: "Not Safe"}
label2id = {"Safe": 0, "Not Safe": 1}
model = AutoModelForSequenceClassification. from_pretrained(model_path,
                                                            num_labels=2,
                                                            id2label=id2label,
                                                            label2id=label2id,)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## freeze all base model parameters
Freezing the base model stops its weights from being updated during training, allowing only the classifier head to learn. This is ideal for small datasets or limited compute, helping prevent overfitting while leveraging pretrained knowledge.

In [ ]:
# freeze all base model parameters
for name, param in model.base_model.named_parameters():
  param. requires_grad = False

In [ ]:
model.base_model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
model.base_model.named_parameters

<bound method Module.named_parameters of BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): BertIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): BertOutput(
          (dense): Linear(in_features=3072, out_features=768, bias=True)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (pooler): BertPooler(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (activation): Tanh()
  )
)>

In [ ]:
model.base_model.named_parameters()

<generator object Module.named_parameters at 0x79d0292ffa40>

In [ ]:
for name, param in model.base_model.named_parameters(name):
  print(name)

pooler.dense.bias.embeddings.word_embeddings.weight
pooler.dense.bias.embeddings.position_embeddings.weight
pooler.dense.bias.embeddings.token_type_embeddings.weight
pooler.dense.bias.embeddings.LayerNorm.weight
pooler.dense.bias.embeddings.LayerNorm.bias
pooler.dense.bias.encoder.layer.0.attention.self.query.weight
pooler.dense.bias.encoder.layer.0.attention.self.query.bias
pooler.dense.bias.encoder.layer.0.attention.self.key.weight
pooler.dense.bias.encoder.layer.0.attention.self.key.bias
pooler.dense.bias.encoder.layer.0.attention.self.value.weight
pooler.dense.bias.encoder.layer.0.attention.self.value.bias
pooler.dense.bias.encoder.layer.0.attention.output.dense.weight
pooler.dense.bias.encoder.layer.0.attention.output.dense.bias
pooler.dense.bias.encoder.layer.0.attention.output.LayerNorm.weight
pooler.dense.bias.encoder.layer.0.attention.output.LayerNorm.bias
pooler.dense.bias.encoder.layer.0.intermediate.dense.weight
pooler.dense.bias.encoder.layer.0.intermediate.dense.bias
pool

## unfreeze base model pooling layers
This code unfreezes only the pooler layer of the transformer, allowing minimal adaptation while keeping the rest frozen. It’s a balanced approach to fine-tuning—enabling slight customization without the cost of full retraining.



In [ ]:
# unfreeze base model pooling layers
for name, param in model.base_model.named_parameters():
    if "pooler" in name:
        param.requires_grad = True

In [ ]:
for name, param in model.base_model.named_parameters():
    if "pooler" in name:
        print(name)

pooler.dense.weight
pooler.dense.bias


## Tokenization

In [ ]:
# define text preprocessing
def preprocess_function(examples) :
  return tokenizer(examples["text"], truncation=True)

# preprocess all datasets
tokenized_data = ddict.map(preprocess_function, batched=True)

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

In [ ]:
# create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizerFast(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

## Evaluation Metrics


In [ ]:
# load metrics
accuracy = evaluate. load("accuracy")
auc_score = evaluate. load("roc_auc")

def compute_metrics(eval_pred):
  # get predictions
  predictions, labels = eval_pred

  # apply softmax to get probabilities
  probabilities = np.exp(predictions) / np.exp(predictions).sum(-1,keepdims=True)

  # use probabilities of the positive class for ROC AUC
  positive_class_probs = probabilities[:, 1]
  # compute auc
  auc = np. round(auc_score.compute(prediction_scores=positive_class_probs, references=labels) ['roc_auc'],3)

  # predict most probable class
  predicted_classes = np.argmax(predictions, axis=1)
  # compute accuracy
  acc = np. round(accuracy. compute(predictions=predicted_classes, references=labels) ['accuracy'],3)

  return {"Accuracy": acc, "AUC": auc}


## Training Arguments


In [ ]:
# hyperparameters
lr = 2e-4
batch_size = 8
num_epochs = 10

training_args = TrainingArguments(
    output_dir="bert-phishing-classifier_teacher",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)


## Training Model




In [ ]:
tokenized_data["test"].filter(lambda x: x["labels"] is None)

Filter:   0%|          | 0/450 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 0
})

In [ ]:
tokenized_data["train"].filter(lambda x: x["labels"] is None)

Filter:   0%|          | 0/2100 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 0
})

In [ ]:
tokenized_data["validation"].filter(lambda x: x["labels"] is None)

Filter:   0%|          | 0/450 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 0
})

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

<ipython-input-28-9d7aa9923850>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Auc
1,0.507600,0.366776,0.856000,0.928000
2,0.407200,0.334130,0.873000,0.935000
3,0.356200,0.325476,0.880000,0.937000
4,0.359100,0.358673,0.862000,0.939000
5,0.351700,0.350271,0.869000,0.943000
6,0.347300,0.309950,0.887000,0.943000
7,0.333600,0.306860,0.880000,0.944000
8,0.310600,0.302860,0.891000,0.945000
9,0.312400,0.302674,0.891000,0.945000
10,0.313600,0.305873,0.884000,0.945000


TrainOutput(global_step=2630, training_loss=0.3599427748995589, metrics={'train_runtime': 138.2313, 'train_samples_per_second': 151.919, 'train_steps_per_second': 19.026, 'total_flos': 706603239165360.0, 'train_loss': 0.3599427748995589, 'epoch': 10.0})

In [ ]:
# apply model to validation dataset
predictions = trainer.predict(tokenized_data["test"])

# Extract the logits and labels from the predictions object
logits = predictions.predictions
labels = predictions. label_ids

# Use your compute_metrics function
metrics = compute_metrics((logits, labels))
print(metrics)


{'Accuracy': np.float64(0.871), 'AUC': np.float64(0.951)}


## Save and Test the Model

In [ ]:
import os
os.getcwd()

'/content'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
trainer.save_model("/content/drive/MyDrive/Phishing_Detection")


In [ ]:
from transformers import pipeline

# Load your phishing detection model
phishing_detector = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/Phishing_Detection",
    return_all_scores=True  # optional: returns scores for both "Safe" and "Not Safe"
)

# Example test
text = "http://freebonus-giftcard.net/verify-now"
result = phishing_detector(text)

print(result)


Device set to use cuda:0


[[{'label': 'Safe', 'score': 0.6910765767097473}, {'label': 'Not Safe', 'score': 0.3089234232902527}]]


/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
text = "http://appleid-support-center.net/reset-password"
result = phishing_detector(text)

print(result)


[[{'label': 'Safe', 'score': 0.17064818739891052}, {'label': 'Not Safe', 'score': 0.8293517827987671}]]
